# Zero-Shot Classification — Detection de Fake News sans entrainement
## Peut-on detecter les fake news sans aucun exemple d'entrainement ?

**Objectif** : Evaluer la capacite de modeles pre-entraines (NLI) a classifier des declarations politiques comme Fake ou Real **sans aucun fine-tuning** sur le LIAR dataset.

**Principe du Zero-Shot** :
- On utilise un modele entraine sur le **Natural Language Inference (NLI)** — une tache qui consiste a determiner si une hypothese decoule d'une premisse
- On reformule notre classification en NLI : "Ce texte est une information vraie" vs "Ce texte est une fausse information"
- Le modele attribue un score de probabilite a chaque hypothese candidate

**Modeles testes** :
1. **BART-large-MNLI** (Facebook) — modele seq2seq entraine sur MultiNLI
2. **DeBERTa-v3-base-MNLI** (Microsoft) — modele NLI compact et performant

**Interet pedagogique** :
- Comparaison Zero-Shot vs modeles supervises (TF-IDF, BERT fine-tune)
- Mesurer l'apport reel du fine-tuning
- Tester differentes formulations de labels (prompt engineering)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

from transformers import pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
COLORS_BIN = ["#e74c3c", "#2ecc71"]

print("Imports OK")

Imports OK


## 1. Chargement des donnees

On utilise le **test set LIAR** (1 267 declarations) et le dataset **FakeNewsNet PolitiFact** (1 056 titres) pour une evaluation complete.

In [2]:
# LIAR test set
df_test = pd.read_parquet(DATA_DIR / "liar_test.parquet")
texts_liar = df_test["statement"].tolist()
labels_liar = df_test["label_binary"].tolist()

# FakeNewsNet PolitiFact
df_fnn = pd.read_parquet(DATA_DIR / "fakenewsnet_politifact.parquet")
texts_fnn = df_fnn["title"].tolist()
labels_fnn = df_fnn["label_binary"].tolist()

print(f"LIAR test      : {len(texts_liar):,} declarations")
print(f"FakeNewsNet    : {len(texts_fnn):,} titres")
print(f"\nLIAR — Fake: {sum(1 for l in labels_liar if l==0)}, Real: {sum(1 for l in labels_liar if l==1)}")
print(f"FNN  — Fake: {sum(1 for l in labels_fnn if l==0)}, Real: {sum(1 for l in labels_fnn if l==1)}")

print("\nExemples LIAR :")
for i in range(3):
    tag = "Fake" if labels_liar[i] == 0 else "Real"
    print(f"  [{tag}] {texts_liar[i][:100]}")

print("\nExemples FakeNewsNet :")
for i in range(3):
    tag = "Fake" if labels_fnn[i] == 0 else "Real"
    print(f"  [{tag}] {texts_fnn[i][:100]}")

LIAR test      : 1,267 declarations
FakeNewsNet    : 1,056 titres

LIAR — Fake: 553, Real: 714
FNN  — Fake: 432, Real: 624

Exemples LIAR :
  [Real] Building a wall on the U.S.-Mexico border will take literally years.
  [Fake] Wisconsin is on pace to double the number of layoffs this year.
  [Fake] Says John McCain has done nothing to help the vets.

Exemples FakeNewsNet :
  [Fake] BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs
  [Fake] Court Orders Obama To Pay $400 Million In Restitution
  [Fake] UPDATE: Second Roy Moore Accuser Works For Michelle Obama Right NOW -


## 2. Principe du Zero-Shot Classification

Le **Zero-Shot** utilise un modele NLI (Natural Language Inference) pour classifier sans exemples d'entrainement.

**Comment ca marche ?**
1. On donne au modele un **texte** (premisse) et des **labels candidats** (hypotheses)
2. Le modele evalue pour chaque label : "Est-ce que le texte **implique** cette hypothese ?"
3. Le label avec le score d'entailment le plus eleve est choisi

**Exemple** :
- Texte : *"The unemployment rate has dropped to 4%"*
- Hypothese 1 : *"This is a true and credible statement"* → score 0.72
- Hypothese 2 : *"This is a false or misleading statement"* → score 0.28
- Prediction : **Real** (score 0.72)

**L'importance du prompt** : La formulation des labels candidats impacte fortement les resultats. On teste plusieurs variantes.

In [3]:
# Differentes formulations de labels candidats a tester
LABEL_SETS = {
    "simple": {
        "candidates": ["fake", "real"],
        "mapping": {"fake": 0, "real": 1},
    },
    "descriptif": {
        "candidates": ["false or misleading statement", "true and credible statement"],
        "mapping": {"false or misleading statement": 0, "true and credible statement": 1},
    },
    "news_framing": {
        "candidates": ["misinformation or fake news", "accurate and factual news"],
        "mapping": {"misinformation or fake news": 0, "accurate and factual news": 1},
    },
}

print("Jeux de labels candidats :")
for name, cfg in LABEL_SETS.items():
    print(f"  {name:15s} → {cfg['candidates']}")

Jeux de labels candidats :
  simple          → ['fake', 'real']
  descriptif      → ['false or misleading statement', 'true and credible statement']
  news_framing    → ['misinformation or fake news', 'accurate and factual news']


## 3. Zero-Shot avec BART-large-MNLI

**BART-large-MNLI** (Lewis et al., 2020) est le modele de reference pour le zero-shot :
- Architecture encodeur-decodeur (406M parametres)
- Fine-tune sur **MultiNLI** (433K paires de phrases, 3 classes : entailment, neutral, contradiction)
- Tres utilise dans la communaute HuggingFace pour le zero-shot

In [4]:
# Fonction utilitaire pour le zero-shot par batch
def zero_shot_predict(classifier, texts, candidate_labels, label_mapping, batch_size=32):
    """
    Applique le zero-shot sur une liste de textes et retourne les predictions binaires.
    """
    predictions = []
    scores_all = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Zero-Shot"):
        batch = texts[i:i + batch_size]
        results = classifier(batch, candidate_labels=candidate_labels, batch_size=batch_size)

        # Si un seul texte, transformer en liste
        if isinstance(results, dict):
            results = [results]

        for res in results:
            top_label = res["labels"][0]
            predictions.append(label_mapping[top_label])
            scores_all.append(res["scores"][0])

    return np.array(predictions), np.array(scores_all)


def evaluate_zero_shot(y_true, y_pred, name):
    """Affiche les metriques et retourne un dict."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    prec = precision_score(y_true, y_pred, average="weighted")
    rec = recall_score(y_true, y_pred, average="weighted")

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}  |  F1 : {f1:.4f}  |  Precision : {prec:.4f}  |  Recall : {rec:.4f}")
    print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))

    return {"accuracy": round(acc, 4), "f1": round(f1, 4),
            "precision": round(prec, 4), "recall": round(rec, 4),
            "cm": confusion_matrix(y_true, y_pred).tolist()}


print("Fonctions utilitaires definies.")

Fonctions utilitaires definies.


In [5]:
# Chargement du modele BART-large-MNLI
print("Chargement de facebook/bart-large-mnli...")
clf_bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
print("Modele BART-large-MNLI charge.")

Chargement de facebook/bart-large-mnli...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Modele BART-large-MNLI charge.


In [ ]:
# Test sur un exemple pour verifier le fonctionnement
example = "The unemployment rate has dropped to its lowest point in 50 years."
for name, cfg in LABEL_SETS.items():
    res = clf_bart(example, candidate_labels=cfg["candidates"])
    print(f"\n[{name}] Texte : '{example[:80]}...'")
    for label, score in zip(res["labels"], res["scores"]):
        print(f"  {label:45s} → {score:.4f}")


[simple] Texte : 'The unemployment rate has dropped to its lowest point in 50 years....'
  real                                          → 0.9575
  fake                                          → 0.0425

[descriptif] Texte : 'The unemployment rate has dropped to its lowest point in 50 years....'
  true and credible statement                   → 0.7584
  false or misleading statement                 → 0.2416

[news_framing] Texte : 'The unemployment rate has dropped to its lowest point in 50 years....'
  accurate and factual news                     → 0.9410
  misinformation or fake news                   → 0.0590


: 

### 3.1 BART-MNLI sur LIAR test — Comparaison des formulations de labels

On teste les 3 formulations de labels candidats sur le test set LIAR pour identifier laquelle fonctionne le mieux.

In [ ]:
# BART-MNLI : tester les 3 formulations sur LIAR
bart_results_liar = {}

for label_name, cfg in LABEL_SETS.items():
    print(f"\n{'='*60}")
    print(f"BART-MNLI — Labels : '{label_name}' → {cfg['candidates']}")
    print(f"{'='*60}")

    preds, scores = zero_shot_predict(
        clf_bart, texts_liar, cfg["candidates"], cfg["mapping"], batch_size=32
    )
    metrics = evaluate_zero_shot(labels_liar, preds, f"BART-MNLI [{label_name}] — LIAR")
    metrics["scores_mean"] = round(scores.mean(), 4)
    metrics["scores_std"] = round(scores.std(), 4)
    bart_results_liar[label_name] = metrics

# Resume
print("\n" + "="*60)
print("RESUME — BART-MNLI sur LIAR (3 formulations)")
print("="*60)
for name, m in bart_results_liar.items():
    print(f"  {name:15s} → Acc: {m['accuracy']:.4f}  F1: {m['f1']:.4f}  (confiance moy: {m['scores_mean']:.4f})")


BART-MNLI — Labels : 'simple' → ['fake', 'real']


Zero-Shot:  52%|█████▎    | 21/40 [18:06<18:59, 59.98s/it]

In [ ]:
# Visualisation : comparaison des formulations (BART sur LIAR)
prompt_compare = pd.DataFrame({
    name: {"Accuracy": m["accuracy"], "F1-Score": m["f1"], "Precision": m["precision"], "Recall": m["recall"]}
    for name, m in bart_results_liar.items()
}).T

fig = px.bar(
    prompt_compare.reset_index().melt(id_vars="index", var_name="Metrique", value_name="Score"),
    x="index", y="Score", color="Metrique", barmode="group",
    title="BART-MNLI sur LIAR — Impact de la formulation des labels",
    labels={"index": "Formulation des labels", "Score": "Score"},
    color_discrete_sequence=["#3498db", "#e74c3c", "#f39c12", "#2ecc71"],
)
fig.update_layout(template="plotly_white", height=450, yaxis=dict(range=[0, 1]))
fig.show()

## 4. Zero-Shot avec DeBERTa-v3-base-MNLI

**DeBERTa-v3** (He et al., 2023) est un modele NLI plus recent et compact :
- 184M parametres (vs 406M pour BART-large)
- Architecture DeBERTa avec disentangled attention
- Excellent rapport performance/taille
- On utilise la meilleure formulation de labels identifiee ci-dessus

In [ ]:
# Chargement DeBERTa-v3-base-MNLI
print("Chargement de cross-encoder/nli-deberta-v3-base...")
clf_deberta = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=-1
)
print("Modele DeBERTa-v3-base-MNLI charge.")

In [ ]:
# DeBERTa : tester les 3 formulations sur LIAR
deberta_results_liar = {}

for label_name, cfg in LABEL_SETS.items():
    print(f"\n{'='*60}")
    print(f"DeBERTa — Labels : '{label_name}' → {cfg['candidates']}")
    print(f"{'='*60}")

    preds, scores = zero_shot_predict(
        clf_deberta, texts_liar, cfg["candidates"], cfg["mapping"], batch_size=32
    )
    metrics = evaluate_zero_shot(labels_liar, preds, f"DeBERTa [{label_name}] — LIAR")
    metrics["scores_mean"] = round(scores.mean(), 4)
    metrics["scores_std"] = round(scores.std(), 4)
    deberta_results_liar[label_name] = metrics

# Resume
print("\n" + "="*60)
print("RESUME — DeBERTa sur LIAR (3 formulations)")
print("="*60)
for name, m in deberta_results_liar.items():
    print(f"  {name:15s} → Acc: {m['accuracy']:.4f}  F1: {m['f1']:.4f}  (confiance moy: {m['scores_mean']:.4f})")

## 5. Evaluation sur FakeNewsNet PolitiFact (Out-of-Domain)

On teste les modeles zero-shot sur un dataset externe pour mesurer leur capacite de generalisation. On utilise la **meilleure formulation** identifiee sur LIAR.

In [ ]:
# Selectionner la meilleure formulation (celle avec le meilleur F1 sur LIAR)
best_bart_prompt = max(bart_results_liar, key=lambda k: bart_results_liar[k]["f1"])
best_deberta_prompt = max(deberta_results_liar, key=lambda k: deberta_results_liar[k]["f1"])

print(f"Meilleure formulation BART    : '{best_bart_prompt}' (F1={bart_results_liar[best_bart_prompt]['f1']:.4f})")
print(f"Meilleure formulation DeBERTa : '{best_deberta_prompt}' (F1={deberta_results_liar[best_deberta_prompt]['f1']:.4f})")

# Evaluation BART sur FakeNewsNet
print(f"\n{'='*60}")
print("BART-MNLI sur FakeNewsNet PolitiFact")
print(f"{'='*60}")
cfg_bart = LABEL_SETS[best_bart_prompt]
preds_bart_fnn, scores_bart_fnn = zero_shot_predict(
    clf_bart, texts_fnn, cfg_bart["candidates"], cfg_bart["mapping"], batch_size=32
)
metrics_bart_fnn = evaluate_zero_shot(labels_fnn, preds_bart_fnn, f"BART-MNLI [{best_bart_prompt}] — FakeNewsNet")

# Evaluation DeBERTa sur FakeNewsNet
print(f"\n{'='*60}")
print("DeBERTa sur FakeNewsNet PolitiFact")
print(f"{'='*60}")
cfg_deb = LABEL_SETS[best_deberta_prompt]
preds_deb_fnn, scores_deb_fnn = zero_shot_predict(
    clf_deberta, texts_fnn, cfg_deb["candidates"], cfg_deb["mapping"], batch_size=32
)
metrics_deb_fnn = evaluate_zero_shot(labels_fnn, preds_deb_fnn, f"DeBERTa [{best_deberta_prompt}] — FakeNewsNet")

## 6. Comparaison globale — Zero-Shot vs Modeles supervises

On compare les resultats zero-shot avec les modeles entraines (TF-IDF, DistilBERT, RoBERTa) pour mesurer l'apport du fine-tuning.

In [ ]:
# Tableau comparatif final — LIAR test set
# Resultats des modeles supervises (issus des notebooks precedents)
supervised_results = {
    "TF-IDF + LogReg": {"accuracy": 0.6006, "f1": 0.6022},
    "TF-IDF + LinearSVC": {"accuracy": 0.6401, "f1": 0.6407},
    "DistilBERT (fine-tuned)": {"accuracy": 0.6425, "f1": 0.6296},
    "RoBERTa (fine-tuned)": {"accuracy": 0.6472, "f1": 0.6423},
}

# Meilleurs resultats zero-shot sur LIAR
best_bart_m = bart_results_liar[best_bart_prompt]
best_deb_m = deberta_results_liar[best_deberta_prompt]

all_results_liar = {
    **supervised_results,
    f"BART-MNLI zero-shot [{best_bart_prompt}]": {
        "accuracy": best_bart_m["accuracy"], "f1": best_bart_m["f1"]
    },
    f"DeBERTa zero-shot [{best_deberta_prompt}]": {
        "accuracy": best_deb_m["accuracy"], "f1": best_deb_m["f1"]
    },
}

compare_df = pd.DataFrame(all_results_liar).T
compare_df.columns = ["Accuracy", "F1-Score"]
compare_df = compare_df.sort_values("F1-Score", ascending=False)

print("=== Comparaison complete sur LIAR test set ===")
print(compare_df.to_string())

# Visualisation
fig = go.Figure()
models = compare_df.index.tolist()
colors_acc = ["#3498db" if "zero-shot" not in m else "#9b59b6" for m in models]
colors_f1 = ["#e74c3c" if "zero-shot" not in m else "#e67e22" for m in models]

fig.add_trace(go.Bar(name="Accuracy", x=models, y=compare_df["Accuracy"], marker_color=colors_acc))
fig.add_trace(go.Bar(name="F1-Score", x=models, y=compare_df["F1-Score"], marker_color=colors_f1))

fig.update_layout(
    barmode="group",
    title="Supervise vs Zero-Shot — Performance sur LIAR test set",
    yaxis_title="Score", template="plotly_white", height=500,
    yaxis=dict(range=[0, 1]),
)
fig.show()

## 7. Analyse des erreurs et distribution des scores de confiance

In [ ]:
# Relancer le meilleur modele BART sur LIAR pour l'analyse d'erreurs
cfg_best = LABEL_SETS[best_bart_prompt]
preds_bart_liar, scores_bart_liar = zero_shot_predict(
    clf_bart, texts_liar, cfg_best["candidates"], cfg_best["mapping"], batch_size=32
)

# Distribution des scores de confiance
df_analysis = pd.DataFrame({
    "text": texts_liar,
    "true_label": labels_liar,
    "pred_label": preds_bart_liar,
    "confidence": scores_bart_liar,
    "correct": np.array(labels_liar) == preds_bart_liar,
})
df_analysis["true_name"] = df_analysis["true_label"].map({0: "Fake", 1: "Real"})
df_analysis["pred_name"] = df_analysis["pred_label"].map({0: "Fake", 1: "Real"})

fig = px.histogram(
    df_analysis, x="confidence", color="correct",
    nbins=40, barmode="overlay", opacity=0.7,
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    title="Distribution des scores de confiance (BART-MNLI) — Correct vs Incorrect",
    labels={"confidence": "Score de confiance", "correct": "Prediction correcte"},
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

print(f"Confiance moyenne — Correct   : {df_analysis[df_analysis['correct']]['confidence'].mean():.4f}")
print(f"Confiance moyenne — Incorrect : {df_analysis[~df_analysis['correct']]['confidence'].mean():.4f}")

In [ ]:
# Exemples d'erreurs — Faux Negatifs (Fake predit Real) et Faux Positifs (Real predit Fake)
fn = df_analysis[(df_analysis["true_label"] == 0) & (df_analysis["pred_label"] == 1)].sort_values("confidence", ascending=False)
fp = df_analysis[(df_analysis["true_label"] == 1) & (df_analysis["pred_label"] == 0)].sort_values("confidence", ascending=False)

print("=== Faux Negatifs (Fake predit comme Real) — Top 5 par confiance ===")
for _, row in fn.head(5).iterrows():
    print(f"  [conf={row['confidence']:.3f}] {row['text'][:120]}")

print(f"\n=== Faux Positifs (Real predit comme Fake) — Top 5 par confiance ===")
for _, row in fp.head(5).iterrows():
    print(f"  [conf={row['confidence']:.3f}] {row['text'][:120]}")

print(f"\nTotal Faux Negatifs : {len(fn)} | Total Faux Positifs : {len(fp)}")

In [ ]:
# Matrices de confusion — BART et DeBERTa sur LIAR
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"BART-MNLI [{best_bart_prompt}]",
        f"DeBERTa [{best_deberta_prompt}]"
    ]
)

for i, (name, results) in enumerate([
    ("BART", bart_results_liar[best_bart_prompt]),
    ("DeBERTa", deberta_results_liar[best_deberta_prompt]),
]):
    cm = np.array(results["cm"])
    fig.add_trace(go.Heatmap(
        z=cm, x=["Fake", "Real"], y=["Fake", "Real"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size": 16},
    ), row=1, col=i+1)

fig.update_yaxes(title_text="Reel", row=1, col=1)
fig.update_xaxes(title_text="Predit")
fig.update_layout(
    title="Matrices de confusion — Zero-Shot sur LIAR test set",
    template="plotly_white", height=400
)
fig.show()

## 8. Synthese et conclusions

### Resultats cles

| Aspect | Observation |
|--------|-------------|
| **Prompt Engineering** | La formulation des labels candidats a un impact significatif sur les resultats. Les labels descriptifs ("false or misleading statement") devraient surpasser les labels simples ("fake/real") |
| **BART vs DeBERTa** | A comparer apres execution — DeBERTa est plus recent et souvent plus performant malgre sa taille inferieure |
| **Zero-Shot vs Supervise** | Le zero-shot devrait atteindre ~50-55% (proche du hasard) sur LIAR, car la tache est intrinsequement difficile sans connaissance factuelle |
| **Out-of-domain** | Le zero-shot pourrait mieux generaliser sur FakeNewsNet car il n'a pas appris de biais specifiques au corpus LIAR |

### Pourquoi le zero-shot est limite pour les fake news ?

1. **Le NLI ne verifie pas les faits** : Le modele evalue la plausibilite linguistique, pas la veracite factuelle. "The GDP grew by 200%" semble grammaticalement correct → predit "real"
2. **Declarations politiques ambigues** : Le LIAR dataset contient des "half-true" et "mostly-true" — des nuances que le zero-shot ne peut pas capturer
3. **Pas de contexte** : Le modele ne connait ni le speaker, ni le contexte politique, ni l'historique de veracite
4. **Textes courts** : Avec seulement ~17 mots en moyenne, le modele a peu de signal

### Apport pedagogique

Le zero-shot est utile comme **baseline non supervisee** :
- Il montre que la detection de fake news necessite un **apprentissage specifique** (fine-tuning)
- Il illustre les limites du **transfer learning** sans adaptation
- Le prompt engineering montre l'importance de la **formulation de la tache**
- La comparaison in-domain vs out-of-domain revele si le modele generalise ou memorise